In [1]:
import sqlite3
from datetime import datetime, timezone

# 設定
DB_PATH = "/app/cat.db"
# 削除境界線: 2026年2月21日 00:00:00 (JST)
cutoff_dt = datetime(2026, 2, 21, 0, 0, 0, tzinfo=timezone.utc)
cutoff_ts = cutoff_dt.timestamp()

def delete_old_data():
    with sqlite3.connect(DB_PATH, timeout=30) as conn:
        # 1. データの件数確認（削除前）
        before_raw = conn.execute("SELECT COUNT(*) FROM raw_data WHERE timestamp < ?", (cutoff_ts,)).fetchone()[0]
        before_labels = conn.execute("SELECT COUNT(*) FROM labels WHERE start_ts < ?", (cutoff_ts,)).fetchone()[0]
        
        print(f"削除対象: 生データ {before_raw} 件 / ラベル {before_labels} 件")
        
        if before_raw == 0 and before_labels == 0:
            print("削除対象のデータはありませんでした。")
            return

        # 2. 削除実行
        conn.execute("DELETE FROM raw_data WHERE timestamp < ?", (cutoff_ts,))
        conn.execute("DELETE FROM labels WHERE start_ts < ?", (cutoff_ts,))
        
        # 3. データベースの最適化（ファイルサイズを小さくする）
        conn.execute("VACUUM")
        
        conn.commit()
        print(f"✅ 2026/02/21 以前のデータを削除し、DBを最適化しました。")

# 実行
delete_old_data()

# 画面を更新するために、メインのUIセルで「REFRESH」ボタンを押すか、
# メインコードの update_ui() をここで呼んでください。


削除対象: 生データ 5443 件 / ラベル 30 件


OperationalError: cannot VACUUM from within a transaction